## JDBC connection setup

In [0]:
jdbc_hostname = "sql-segrid-anuj.database.windows.net"
jdbc_port = 1433
jdbc_database = "EnergyGridDB"
jdbc_username = "sqlamin"  
jdbc_password = "SQL-Password"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database}"

connection_props = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col, lit, avg, when as w, window, count

In [0]:
storage_account_name = "segrid"
storage_account_key = "STORAGE_ACCOUNT_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
curated_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/energy_readings/"
df_curated = spark.read.parquet(curated_path)

df_fact = (
    df_curated
    .withColumn("meter_id", lit("HOUSE_001"))
    .select(
        "meter_id", "reading_timestamp",
        "global_active_power_kw", "global_reactive_power_kw", "voltage_v", "global_intensity_a",
        "sub_metering_kitchen_wh", "sub_metering_laundry_wh", "sub_metering_waterheater_ac_wh",
        "unmetered_power_wh", "is_weekend", "ingestion_source", "ingestion_timestamp"
    )
)

In [0]:
(
    df_fact.write
    .mode("append")
    .jdbc(url=jdbc_url, table="fact_energy_readings", properties=connection_props)
)
print("fact_energy_readings loaded.")

fact_energy_readings loaded.


In [0]:
from pyspark.sql.functions import avg, when as w

df_patterns = (
    df_curated
    .withColumn("day_type", w(col("is_weekend"), "Weekend").otherwise("Weekday"))
    .groupBy("day_type", "hour_of_day")
    .agg(
        avg("global_active_power_kw").alias("avg_global_active_power_kw"),
        avg("sub_metering_kitchen_wh").alias("avg_kitchen_wh"),
        avg("sub_metering_laundry_wh").alias("avg_laundry_wh"),
        avg("sub_metering_waterheater_ac_wh").alias("avg_waterheater_ac_wh"),
    )
)

(
    df_patterns.write
    .mode("overwrite")
    .jdbc(url=jdbc_url, table="dbo.ConsumptionPatterns", properties=connection_props)
)
print(f"ConsumptionPatterns loaded: {df_patterns.count()} rows")

ConsumptionPatterns loaded: 48 rows


In [0]:
df_anomalies_out = (
    df_curated
    .filter(col("anomaly_type").isNotNull())
    .select("reading_timestamp", "anomaly_type", "voltage_v",
            "global_reactive_power_kw", "global_intensity_a", "severity")
)

(
    df_anomalies_out.write
    .mode("overwrite")
    .jdbc(url=jdbc_url, table="dbo.PowerQualityAnomalies", properties=connection_props)
)
print(f"PowerQualityAnomalies loaded: {df_anomalies_out.count()} rows")

PowerQualityAnomalies loaded: 0 rows
